# 6.3.2 Tool Use and Function Calling

Build a function registry with schema validation and simulated tool call dispatch.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)

REGISTRY = {}

def register(name, desc, params):
    def dec(fn):
        REGISTRY[name] = {'fn': fn, 'desc': desc, 'params': params}
        return fn
    return dec

@register('get_weather', 'Get weather', {'city': {'type': 'string', 'required': True}})
def get_weather(city):
    return {'city': city, 'temp': 20, 'unit': 'C'}

@register('calculate', 'Math eval', {'expr': {'type': 'string', 'required': True}})
def calculate(expr):
    try: return {'result': eval(expr, {'__builtins__': {}}, {})}
    except: return {'error': 'eval failed'}

def dispatch(name, args):
    if name not in REGISTRY: return {'error': f'Unknown: {name}'}
    for p, m in REGISTRY[name]['params'].items():
        if m.get('required') and p not in args:
            return {'error': f'Missing: {p}'}
    return REGISTRY[name]['fn'](**args)

print(dispatch('get_weather', {'city': 'Paris'}))
print(dispatch('calculate', {'expr': '2**10'}))
print(dispatch('calculate', {}))  # missing required
print(dispatch('unknown', {}))    # not registered

In [ ]:
calls = [
    ('get_weather', {'city': 'Paris'}),
    ('calculate', {'expr': '3*7'}),
    ('calculate', {}),
    ('missing_fn', {}),
]
results = [dispatch(n, a) for n, a in calls]
success = [1 if 'error' not in r else 0 for r in results]
labels = [f'{n}({list(a.keys())})' for n, a in calls]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(labels, success, color=['mediumseagreen' if s else 'tomato' for s in success])
ax.set(xlabel='Success', title='Tool Call Validation')
ax.set_xlim(0, 1.4); ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('output/tool_use.png')
print('Saved tool_use.png')